In [2]:
from src.utils import preprocess_boxes, calculate_pixel_value, horizontal_pixel_pos, horizontal_pos_conversion, predict_distance, get_image_size, resize_to_x_by_x, prepare_feature_vector
from src.model_initialization import load_yolo_model, load_posenet_model, load_depth_estimation_model, load_orientation_model, load_depth_curve_spec
from src.simulation_a_star import simulation, animate_cost_map, create_internal_obstacles
from src.direction_speed import  calculate_speed
from src.depth_estimation import estimate_depth
from src.direction_speed import calculate_direction, calculate_speed
from src.simulation_a_star import animate_cost_map, simulation, create_internal_obstacles
from src.draw_boxes import detect_persons
from src.utils import preprocess_boxes, calculate_pixel_value, horizontal_pixel_pos, horizontal_pos_conversion, predict_distance, get_image_size, resize_to_x_by_x

2024-08-23 09:50:43.633516: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.1 SSE4.2
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


ModuleNotFoundError: No module named 'joblib'

In [ ]:
import numpy as np

SCALE = 0.1

In [ ]:
yolo_model = load_yolo_model()
posenet_model = load_posenet_model()
orientation_model = load_orientation_model()
depth_processor, depth_model = load_depth_estimation_model()

best_fitting_depth_curve_type, best_fitting_depth_curve_param  = load_depth_curve_spec()

In [ ]:
# CHANGE to take a random image from your depth set to test its calibration
input_img = "/Users/paulrichard/Documents/HAAN/data/input/random_test/IMG_2346.jpeg"

In [ ]:
DISP = True
ROBOT_SPEED = 1.11*SCALE
NEW_SIZE = (1280, 1280)
START = (int(SCALE*0),int(SCALE*0))
GOAL = (int(-500*SCALE),int(900*SCALE))
SIMULATION_TIME = 100000
DT = 100
GRID_SIZE = (1280,1000)

In [ ]:
# Pre-processing of the image to our working format 
img, size_img = get_image_size(input_img)
resized_image = resize_to_x_by_x(img, NEW_SIZE)

# Person detection
coords_box_in_pixel = detect_persons(resized_image, yolo_model)

# Depth Estimation
depth_image = estimate_depth(resized_image, depth_processor, depth_model)
box_without_background = preprocess_boxes(depth_image, coords_box_in_pixel)
mean_pixel_values = calculate_pixel_value(box_without_background)
depth_values = [predict_distance(pv, best_fitting_depth_curve_param, best_fitting_depth_curve_type.lower()) for pv in mean_pixel_values]

# Horizontal position Estimation
horizontal_pos_in_pixel = horizontal_pixel_pos(coords_box_in_pixel)
horizontal_pos_in_cm = horizontal_pos_conversion(horizontal_pos_in_pixel)

# Direction and Speed Estimation
features_vector = [prepare_feature_vector(resized_image, box, posenet_model, depth) for box, depth in zip(coords_box_in_pixel, depth_values)]
directions = [orientation_model.predict(features) for features in features_vector]
directions_in_radians = [np.radians(direction[0]) for direction in directions]
#directions_in_radians[0]= np.pi*3/2 
speeds = calculate_speed(resized_image, coords_box_in_pixel)

# Creating the obstacles for Simulation
num_obstacles = len(coords_box_in_pixel) 
dynamic_obstacles_pos = np.zeros((num_obstacles, 2)) 
dynamic_obstalcles_dir_speed = []  
for i, (horizontal_pos, depth, direction, speed) in enumerate(zip(horizontal_pos_in_cm, depth_values, directions_in_radians, speeds)):
    x_position = int(horizontal_pos*SCALE)
    y_position = int(depth*SCALE)
    
    dynamic_obstacles_pos[i, :] = [x_position, y_position]
    dynamic_obstalcles_dir_speed.append((direction, speed))

In [ ]:
num_obstacles = 20
dynamic_obstacles_pos, dynamic_obstalcles_dir_speed = create_internal_obstacles(num_obstacles)

In [ ]:
#Simulation:
robot_positions, cost_map_times, dynamic_obstacles_times, path_times, dynamic_obstalcles_dir_speed_times= simulation(   dynamic_obstacles_pos, 
                                                                                                                        dynamic_obstalcles_dir_speed, 
                                                                                                                        START, 
                                                                                                                        GOAL, 
                                                                                                                        ROBOT_SPEED, 
                                                                                                                        SIMULATION_TIME, 
                                                                                                                        DT
                                                                                                                    )

In [ ]:
# Animate the simulation results
anim = animate_cost_map(cost_map_times, robot_positions, dynamic_obstacles_times, dynamic_obstalcles_dir_speed_times, GOAL, SIMULATION_TIME, DT, path_times)

# Display the animation
from IPython.display import HTML
HTML(anim.to_jshtml())

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, writers

# Set up the writer for saving as MP4
Writer = writers['ffmpeg']
writer = Writer(fps=15, metadata=dict(artist='Me'), bitrate=1800)

# Save the animation as an MP4 file
# CHANGE for where to save your animation
anim.save('/Users/paulrichard/Documents/HAAN/animation2.mp4', writer=writer)

----
----
## Models initialisation: 


In [5]:
yolo_model = load_yolo_model()
posenet_model = load_posenet_model()
depth_processor, depth_model = load_depth_estimation_model()


SyntaxError: invalid syntax (1939236301.py, line 4)


----
----
## Calibration:

Here is the method we used to find best fitting curve

In [6]:
%matplotlib inline

input_folder = '/Users/prichard/miniconda3/envs/infosys/cv_project/data/input/calibration/normal_size'
output_folder = '/Users/prichard/miniconda3/envs/infosys/cv_project/data/output'
#best_model, prediction_param = process_and_plot_calibration(input_folder, yolo_model, depth_processor, depth_model, output_folder)


Small image error = 0.2629 \
Medium image error = 0.2126 \
High image error = 0.2082 \
Normal image error = 0.1980


![Curve Data](images/Curve_Data_Big1_512.png)
![Curve Data](images/Curve_Data_Big1_1024.png)
![Curve Data](images/Curve_Data_Big1_2048.png)
![Curve Data](images/Curve_Data_Big1_3000.png)


In [7]:
test_image_path = "data/input/calibration/high_size/0.9.jpeg"
DISP = True

robot_speed = 1.11 #  [m/s]

new_size = (1280, 1280)

img, size_img = get_image_size(test_image_path)

resized_image = resize_to_x_by_x(img, new_size)

depth_image = estimate_depth(resized_image, depth_processor, depth_model) # 0.5
coords_box_in_pixel = detect_persons(resized_image, yolo_model) # traduire dans la nouvelle map et pour l'instant mettre tout en cm avec juste pixel = 0.1 cm

filtered_depth_images = preprocess_boxes(depth_image, coords_box_in_pixel)
pixel_values = calculate_pixel_value(filtered_depth_images)

horizontal_pos_in_pixel = horizontal_pixel_pos(coords_box_in_pixel)
horizontal_pos_in_cm = horizontal_pos_conversion(horizontal_pos_in_pixel) # here we change the coordinates st the 0,0 is the robot position 

depth_values = [predict_distance(pv, prediction_param, best_model.lower()) for pv in pixel_values]


directions = calculate_direction(resized_image, coords_box_in_pixel, depth_values, posenet_model, DISP)
speeds = calculate_speed(resized_image, coords_box_in_pixel)


[ WARN:0@1630.572] global /private/var/folders/k1/30mswbxs7r1g6zwn8y4fyt500000gp/T/abs_11nitadzeg/croot/opencv-suite_1691620374638/work/modules/imgcodecs/src/loadsave.cpp (239) findDecoder imread_('data/input/calibration/high_size/0.9.jpeg'): can't open/read file: check file path/integrity


FileNotFoundError: Image not found at data/input/calibration/high_size/0.9.jpeg